# 第二步：ChemVista Novel/Transfer选择与无泄漏拆分冻结

本 Notebook 使用第一步的化合物级结果，固定生成三类角色：

- **ChemVista-Novel**：化合物IK2D从监督开发集完全排除，用于未知化合物泛化验证；
- **ChemVista-Transfer**：公共谱图可用于训练，但ChemVista谱图不训练，仅用于跨实验条件稳定性验证；
- **Development**：用于训练、内部验证和结构分组交叉验证。

拆分只使用预先声明的标签覆盖、结构骨架和谱图元数据，不读取任何模型结果。生成后通过名单哈希冻结。

In [ ]:
from __future__ import annotations

import hashlib
import json
import pickle
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import MACCSkeys, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.ML.Cluster import Butina

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 200)

def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / '性质数据').exists() and (candidate / '质谱数据').exists():
            return candidate
    raise FileNotFoundError('未找到项目根目录。')

def as_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.lower().isin(['true', '1', 'yes'])

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def sha256_values(values) -> str:
    payload = '\n'.join(sorted(map(str, values))) + '\n'
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()

ROOT = find_project_root(Path.cwd())
STEP1_DIR = ROOT / 'outputs' / 'pbt_msms_v1' / 'preparation_step1'
OUTPUT_DIR = ROOT / 'outputs' / 'pbt_msms_v1' / 'split_v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
NOVEL_TOTAL_TARGET = 240
N_CV_FOLDS = 5

print('ROOT:', ROOT)
print('OUTPUT:', OUTPUT_DIR)
print('Python:', sys.executable)
print('Python version:', sys.version.split()[0])
print('pandas:', pd.__version__, 'numpy:', np.__version__, 'RDKit:', rdBase.rdkitVersion)

## 1. 读取第一步固定输出并声明拆分约束

Novel总量固定为240：包含全部38个天然Novel，再从公共重叠候选中选择202个。任务配额按Novel全集计算。

In [ ]:
INPUTS = {
    'compound_master': STEP1_DIR / 'compound_master_modeling.csv',
    'chemvista_feasibility': STEP1_DIR / 'chemvista_split_feasibility.csv',
    'step1_audit': STEP1_DIR / 'step1_audit.json',
    'step1_qc': STEP1_DIR / 'step1_qc_checks.csv',
}
missing = [str(p) for p in INPUTS.values() if not p.exists()]
assert not missing, '请先完整运行第一步Notebook。缺少：\n' + '\n'.join(missing)

compound_master = pd.read_csv(INPUTS['compound_master'], low_memory=False)
chem = pd.read_csv(INPUTS['chemvista_feasibility'], low_memory=False)
step1_qc = pd.read_csv(INPUTS['step1_qc'])
with INPUTS['step1_audit'].open('r', encoding='utf-8') as handle:
    step1_audit = json.load(handle)

assert step1_qc['status'].eq('PASS').all(), '第一步QC未全部通过。'
assert chem['inchikey_2d'].is_unique
assert compound_master['inchikey_2d'].is_unique

P_TASK = 'P_screen_not_ready_binary'
REGRESSION_TASKS = [
    'B_LOGBCF_REGRESSION',
    'P_READY_BOD_D28_OECD301C_SOURCE_301C',
    'T_EC10_MG_L_REGRESSION',
    'T_EC50_MG_L_REGRESSION',
    'T_LC50_MG_L_REGRESSION',
    'T_NOEC_MG_L_REGRESSION',
]
PRIMARY_TASKS = [
    P_TASK, 'B_LOGBCF_REGRESSION',
    'T_EC50_MG_L_REGRESSION', 'T_LC50_MG_L_REGRESSION', 'T_NOEC_MG_L_REGRESSION',
]

REQUIREMENTS = {
    'P_positive': 30,
    'P_negative': 30,
    'B_LOGBCF_REGRESSION': 40,
    'P_READY_BOD_D28_OECD301C_SOURCE_301C': 20,
    'T_EC10_MG_L_REGRESSION': 20,
    'T_EC50_MG_L_REGRESSION': 80,
    'T_LC50_MG_L_REGRESSION': 80,
    'T_NOEC_MG_L_REGRESSION': 80,
}
QUARTILE_REQUIREMENTS = {
    'B_LOGBCF_REGRESSION': 6,
    'P_READY_BOD_D28_OECD301C_SOURCE_301C': 3,
    'T_EC10_MG_L_REGRESSION': 3,
    'T_EC50_MG_L_REGRESSION': 12,
    'T_LC50_MG_L_REGRESSION': 12,
    'T_NOEC_MG_L_REGRESSION': 12,
}

print('第一步版本:', step1_audit['analysis_version'])
print('ChemVista建模候选:', len(chem))
print('天然Novel:', int(as_bool(chem['natural_chemvista_novel']).sum()))
print('公共重叠候选:', int((~as_bool(chem['natural_chemvista_novel'])).sum()))

## 2. 构造任务覆盖与回归值域标记

除总任务数量外，每个回归任务四个四分位区间都设置最低覆盖，避免外测集只包含中间值。

In [ ]:
flags: dict[str, pd.Series] = {}
flags['P_positive'] = pd.to_numeric(chem[f'label__{P_TASK}'], errors='coerce').eq(1)
flags['P_negative'] = pd.to_numeric(chem[f'label__{P_TASK}'], errors='coerce').eq(0)

all_requirements = dict(REQUIREMENTS)
quartile_assignments = {}
for task in REGRESSION_TASKS:
    col = f'label__{task}'
    values = pd.to_numeric(chem[col], errors='coerce')
    flags[task] = values.notna()
    valid = values.dropna()
    # rank(method='first')使相同数值也能确定性分配，避免qcut边界重复失败
    quartile = pd.Series(np.nan, index=chem.index, dtype=float)
    quartile.loc[valid.index] = pd.qcut(valid.rank(method='first'), 4, labels=False).astype(float)
    quartile_assignments[task] = quartile
    for q in range(4):
        key = f'{task}__Q{q + 1}'
        flags[key] = quartile.eq(q)
        all_requirements[key] = QUARTILE_REQUIREMENTS[task]

requirement_table = pd.DataFrame([
    {'criterion': key, 'minimum_in_novel': value, 'available_in_chemvista': int(flags[key].sum())}
    for key, value in all_requirements.items()
])
assert (requirement_table['available_in_chemvista'] >= requirement_table['minimum_in_novel']).all()
requirement_table

## 3. 固定种子贪心选择ChemVista-Novel

全部天然Novel强制入选。其余候选按未满足任务配额、P类别覆盖、回归四分位覆盖和新结构骨架进行确定性评分。哈希只用于完全同分时打破平局。

In [ ]:
natural_mask = as_bool(chem['natural_chemvista_novel'])
selected_indices = set(chem.index[natural_mask].tolist())
remaining_indices = set(chem.index[~natural_mask].tolist())
selected_scaffolds = set(chem.loc[list(selected_indices), 'murcko_scaffold_generic'].dropna().astype(str))

def selected_count(flag_name: str) -> int:
    if not selected_indices:
        return 0
    return int(flags[flag_name].loc[list(selected_indices)].sum())

def deterministic_tie(ik2d: str) -> float:
    digest = hashlib.sha256(f'{SEED}|{ik2d}'.encode('utf-8')).hexdigest()
    return int(digest[:12], 16) / float(16 ** 12)

selection_trace = []
while len(selected_indices) < NOVEL_TOTAL_TARGET:
    deficits = {name: max(0, minimum - selected_count(name)) for name, minimum in all_requirements.items()}
    best_index = None
    best_score = -np.inf
    best_contributions = []
    for idx in remaining_indices:
        contributions = [name for name in all_requirements if deficits[name] > 0 and bool(flags[name].loc[idx])]
        score = 0.0
        for name in contributions:
            weight = 5.0 if name in {'P_positive', 'P_negative'} else 1.0
            score += weight * deficits[name] / max(1, all_requirements[name])
        scaffold = str(chem.loc[idx, 'murcko_scaffold_generic'])
        if scaffold and scaffold != 'nan' and scaffold != 'ACYCLIC' and scaffold not in selected_scaffolds:
            score += 0.25
        score += deterministic_tie(chem.loc[idx, 'inchikey_2d']) * 1e-6
        if score > best_score:
            best_index = idx
            best_score = score
            best_contributions = contributions
    if best_index is None:
        raise RuntimeError('Novel选择提前终止。')
    selected_indices.add(best_index)
    remaining_indices.remove(best_index)
    selected_scaffolds.add(str(chem.loc[best_index, 'murcko_scaffold_generic']))
    selection_trace.append({
        'selection_order': len(selected_indices),
        'inchikey_2d': chem.loc[best_index, 'inchikey_2d'],
        'score_at_selection': best_score,
        'unmet_criteria_contributed': ';'.join(best_contributions),
    })

novel = chem.loc[sorted(selected_indices)].copy()
novel['chemvista_evaluation_role'] = np.where(as_bool(novel['natural_chemvista_novel']), 'natural_novel', 'selected_novel')
transfer = chem.loc[sorted(remaining_indices)].copy()
transfer['chemvista_evaluation_role'] = 'transfer'
selection_trace = pd.DataFrame(selection_trace)

coverage_rows = []
for name, minimum in all_requirements.items():
    observed = int(flags[name].loc[novel.index].sum())
    coverage_rows.append({'criterion': name, 'minimum': minimum, 'observed': observed, 'status': 'PASS' if observed >= minimum else 'FAIL'})
coverage = pd.DataFrame(coverage_rows)
assert coverage['status'].eq('PASS').all(), coverage[coverage['status'].eq('FAIL')]

print('Novel总数:', len(novel))
print('  天然Novel:', int(novel['chemvista_evaluation_role'].eq('natural_novel').sum()))
print('  选择Novel:', int(novel['chemvista_evaluation_role'].eq('selected_novel').sum()))
print('Transfer:', len(transfer))
print('Novel结构骨架数:', novel['murcko_scaffold_generic'].nunique(dropna=True))
coverage[~coverage['criterion'].str.contains('__Q')]

## 4. 生成监督角色与ChemVista评价角色

一个Transfer化合物在监督轴上属于Development，在ChemVista评价轴上属于Transfer。用两个字段避免把两种科学问题混为一个互斥标签。

In [ ]:
novel_keys = set(novel['inchikey_2d'])
transfer_keys = set(transfer['inchikey_2d'])
has_public = as_bool(compound_master['has_public_spectrum'])
has_chem = as_bool(compound_master['has_chemvista_spectrum'])

manifest = compound_master.copy()
manifest['supervised_compound_role'] = np.select(
    [manifest['inchikey_2d'].isin(novel_keys), has_public],
    ['external_novel', 'development'],
    default='property_without_public_spectrum',
)
manifest['chemvista_evaluation_role'] = np.select(
    [manifest['inchikey_2d'].isin(set(novel.loc[novel['chemvista_evaluation_role'].eq('natural_novel'), 'inchikey_2d'])),
     manifest['inchikey_2d'].isin(set(novel.loc[novel['chemvista_evaluation_role'].eq('selected_novel'), 'inchikey_2d'])),
     manifest['inchikey_2d'].isin(transfer_keys)],
    ['natural_novel', 'selected_novel', 'transfer'],
    default='none',
)
development = manifest[manifest['supervised_compound_role'].eq('development')].copy()
external_novel = manifest[manifest['supervised_compound_role'].eq('external_novel')].copy()

assert set(development['inchikey_2d']).isdisjoint(set(external_novel['inchikey_2d']))
assert set(transfer_keys).issubset(set(development['inchikey_2d']))
assert novel_keys.isdisjoint(transfer_keys)

print('Development:', len(development))
print('External Novel:', len(external_novel))
print('Transfer（也是Development化合物）:', len(transfer_keys))
manifest[['supervised_compound_role', 'chemvista_evaluation_role']].value_counts().to_frame('n_compounds')

## 5. Development内部Morgan–Butina结构簇分组五折

全部Development化合物使用Morgan ECFP4（radius=2、2048 bit、无手性）进行Butina结构聚类，再按完整结构簇、任务可用性和P正负类进行确定性贪心折分配。标准Murcko scaffold只保留作审计，不参与fold定义。

In [ ]:
development = development.copy()
development['cv_group'] = ''
development['murcko_scaffold_standard'] = ''
for idx, smiles in development['canonical_smiles'].items():
    mol = Chem.MolFromSmiles(str(smiles)) if pd.notna(smiles) else None
    if mol is None:
        continue
    scaffold_mol = MurckoScaffold.GetScaffoldForMol(mol)
    if scaffold_mol.GetNumAtoms() > 0:
        development.loc[idx, 'murcko_scaffold_standard'] = Chem.MolToSmiles(scaffold_mol, isomericSmiles=False)

# CV使用保留原子和键类型的标准Bemis–Murcko骨架。
# 通用骨架会把大量不同芳香性/杂原子骨架压成同一个环系，导致单折异常庞大。
# 采用全体化合物的Morgan结构簇做CV分组（Butina距离阈值0.6，即中心相似度约0.4）。
# 标准Murcko骨架保留作审计；仅按骨架分组会把360个单苯环化合物塞入同一折。
fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048, includeChirality=False)
valid_structure_indices = []
structure_fps = []
for idx, smiles in development['canonical_smiles'].items():
    mol = Chem.MolFromSmiles(str(smiles)) if pd.notna(smiles) else None
    if mol is None:
        development.loc[idx, 'cv_group'] = 'NO_STRUCTURE:' + development.loc[idx, 'inchikey_2d']
    else:
        valid_structure_indices.append(idx)
        structure_fps.append(fpgen.GetFingerprint(mol))

distances = []
for i in range(1, len(structure_fps)):
    similarities = DataStructs.BulkTanimotoSimilarity(structure_fps[i], structure_fps[:i])
    distances.extend([1.0 - value for value in similarities])
clusters = Butina.ClusterData(
    distances, len(structure_fps), 0.6, isDistData=True, reordering=True
) if structure_fps else []
for cluster_id, members in enumerate(clusters):
    for member in members:
        development.loc[valid_structure_indices[member], 'cv_group'] = f'MORGAN_CLUSTER:{cluster_id:04d}'
assert development['cv_group'].ne('').all()

balance_columns = [f'label__{P_TASK}'] + [f'label__{task}' for task in REGRESSION_TASKS]
feature_frame = pd.DataFrame(index=development.index)
feature_frame['total'] = 1.0
feature_frame['P_positive'] = pd.to_numeric(development[f'label__{P_TASK}'], errors='coerce').eq(1).astype(float)
feature_frame['P_negative'] = pd.to_numeric(development[f'label__{P_TASK}'], errors='coerce').eq(0).astype(float)
for task in REGRESSION_TASKS:
    feature_frame[task] = pd.to_numeric(development[f'label__{task}'], errors='coerce').notna().astype(float)
group_features = feature_frame.assign(cv_group=development['cv_group']).groupby('cv_group').sum()
target_per_fold = group_features.sum(axis=0) / N_CV_FOLDS
normalized_size = group_features.div(target_per_fold.replace(0, 1), axis=1).max(axis=1)
group_order = sorted(group_features.index, key=lambda g: (-normalized_size.loc[g], -group_features.loc[g, 'total'], g))
fold_totals = pd.DataFrame(0.0, index=range(N_CV_FOLDS), columns=group_features.columns)
group_to_fold = {}
for group_index, group in enumerate(group_order):
    row = group_features.loc[group]
    if group_index < N_CV_FOLDS:
        # 先保证每个fold至少拥有一个最大结构组
        chosen_fold = group_index
    else:
        scores = {}
        for fold in range(N_CV_FOLDS):
            trial = fold_totals.copy()
            trial.loc[fold] += row
            normalized_error = trial.sub(target_per_fold, axis=1).div(target_per_fold.replace(0, 1), axis=1)
            scores[fold] = float(normalized_error.pow(2).to_numpy().sum())
        chosen_fold = min(scores, key=lambda f: (scores[f], fold_totals.loc[f, 'total'], f))
    group_to_fold[group] = chosen_fold
    fold_totals.loc[chosen_fold] += row
development['cv_fold'] = development['cv_group'].map(group_to_fold).astype(int)
manifest['cv_group'] = manifest['inchikey_2d'].map(development.set_index('inchikey_2d')['cv_group'])
manifest['cv_fold'] = manifest['inchikey_2d'].map(development.set_index('inchikey_2d')['cv_fold']).astype('Int64')

fold_summary = []
for fold, frame in development.groupby('cv_fold'):
    row = {'cv_fold': int(fold), 'n_compounds': len(frame), 'n_groups': frame['cv_group'].nunique()}
    pvals = pd.to_numeric(frame[f'label__{P_TASK}'], errors='coerce')
    row.update({'P_positive': int(pvals.eq(1).sum()), 'P_negative': int(pvals.eq(0).sum())})
    for task in REGRESSION_TASKS:
        row[task] = int(pd.to_numeric(frame[f'label__{task}'], errors='coerce').notna().sum())
    fold_summary.append(row)
fold_summary = pd.DataFrame(fold_summary).sort_values('cv_fold')
assert len(fold_summary) == N_CV_FOLDS
assert (fold_summary[['P_positive', 'P_negative']] > 0).all().all()
fold_summary

# 结构适用域与固定表示碰撞审计。表示完全相同不等于身份泄漏，但必须在外测时单独报告。
development_fp_rows = list(development[['inchikey_2d', 'canonical_smiles', 'cv_fold']].itertuples(index=False))
development_morgan = [fpgen.GetFingerprint(Chem.MolFromSmiles(str(row.canonical_smiles))) for row in development_fp_rows]
development_maccs = [MACCSkeys.GenMACCSKeys(Chem.MolFromSmiles(str(row.canonical_smiles))) for row in development_fp_rows]
novel_applicability_rows = []
for row in novel[['inchikey_2d', 'canonical_smiles', 'chemvista_evaluation_role']].itertuples(index=False):
    mol = Chem.MolFromSmiles(str(row.canonical_smiles))
    fp_morgan = fpgen.GetFingerprint(mol)
    fp_maccs = MACCSkeys.GenMACCSKeys(mol)
    morgan_similarities = np.asarray(DataStructs.BulkTanimotoSimilarity(fp_morgan, development_morgan))
    maccs_similarities = np.asarray(DataStructs.BulkTanimotoSimilarity(fp_maccs, development_maccs))
    closest_index = int(np.argmax(morgan_similarities))
    novel_applicability_rows.append({
        'inchikey_2d': row.inchikey_2d,
        'chemvista_evaluation_role': row.chemvista_evaluation_role,
        'max_morgan_tanimoto_to_development': float(morgan_similarities[closest_index]),
        'closest_development_ik2d': development_fp_rows[closest_index].inchikey_2d,
        'exact_morgan_representation_collision': bool(np.isclose(morgan_similarities, 1.0).any()),
        'exact_maccs_representation_collision': bool(np.isclose(maccs_similarities, 1.0).any()),
    })
novel_applicability = pd.DataFrame(novel_applicability_rows).sort_values('inchikey_2d')
development_cv_applicability_rows = []
development_folds = development['cv_fold'].to_numpy(dtype=int)
for index, row in enumerate(development_fp_rows):
    train_indices = np.flatnonzero(development_folds != int(row.cv_fold))
    similarities = np.asarray(DataStructs.BulkTanimotoSimilarity(development_morgan[index], [development_morgan[j] for j in train_indices]))
    best_local = int(np.argmax(similarities))
    closest_index = int(train_indices[best_local])
    development_cv_applicability_rows.append({
        'inchikey_2d': row.inchikey_2d, 'cv_fold': int(row.cv_fold),
        'max_morgan_tanimoto_to_outer_train': float(similarities[best_local]),
        'closest_outer_train_ik2d': development_fp_rows[closest_index].inchikey_2d,
    })
development_cv_applicability = pd.DataFrame(development_cv_applicability_rows).sort_values('inchikey_2d')
print('Novel Morgan完全表示碰撞:', int(novel_applicability['exact_morgan_representation_collision'].sum()))
print('Novel MACCS完全表示碰撞:', int(novel_applicability['exact_maccs_representation_collision'].sum()))
display(novel_applicability['max_morgan_tanimoto_to_development'].describe(percentiles=[0.1, 0.5, 0.9, 0.95]))

## 6. 拆分统计、泄漏检查与冻结输出

In [ ]:
development = manifest[manifest['supervised_compound_role'].eq('development')].copy()
external_novel = manifest[manifest['supervised_compound_role'].eq('external_novel')].copy()
task_rows = []
for task in PRIMARY_TASKS + [t for t in REGRESSION_TASKS if t not in PRIMARY_TASKS]:
    col = f'label__{task}'
    for subset_name, frame in [
        ('development', development),
        ('chemvista_novel', novel),
        ('chemvista_transfer', transfer),
    ]:
        values = pd.to_numeric(frame[col], errors='coerce').dropna()
        row = {'task_name': task, 'subset': subset_name, 'n_compounds': len(values)}
        if task == P_TASK:
            row.update({'positive': int(values.eq(1).sum()), 'negative': int(values.eq(0).sum()), 'positive_rate': float(values.eq(1).mean())})
        task_rows.append(row)
task_split_summary = pd.DataFrame(task_rows)

dev_keys = set(development['inchikey_2d'])
external_keys = set(external_novel['inchikey_2d'])
checks = []
def add_check(check_id, description, passed, details=''):
    checks.append({'check_id': check_id, 'description': description, 'status': 'PASS' if passed else 'FAIL', 'details': details})

add_check('SPLIT001', 'Novel总量等于预注册目标', len(novel) == NOVEL_TOTAL_TARGET, f'{len(novel)} / {NOVEL_TOTAL_TARGET}')
add_check('SPLIT002', '全部天然Novel均进入Novel', set(chem.loc[natural_mask, 'inchikey_2d']).issubset(novel_keys), str(int(natural_mask.sum())))
add_check('SPLIT003', '监督Development与External Novel的IK2D无交集', dev_keys.isdisjoint(external_keys), str(len(dev_keys & external_keys)))
add_check('SPLIT004', 'Novel与Transfer无交集', novel_keys.isdisjoint(transfer_keys), str(len(novel_keys & transfer_keys)))
add_check('SPLIT005', 'Transfer全部属于监督Development', transfer_keys.issubset(dev_keys), f'{len(transfer_keys)} / {len(transfer_keys)}')
add_check('SPLIT006', '所有Novel任务覆盖约束通过', coverage['status'].eq('PASS').all())
add_check('SPLIT007', '每个Development CV fold均有P正负类', (fold_summary[['P_positive', 'P_negative']] > 0).all().all())
group_fold_counts = development.groupby('cv_group')['cv_fold'].nunique()
add_check('SPLIT008', '同一cv_group只属于一个fold', group_fold_counts.le(1).all(), str(int((group_fold_counts > 1).sum())))
add_check('SPLIT009', '正式P/vP/T未用于Novel选择约束', not any('formal' in key for key in all_requirements))
fold_size_ratio = float(fold_summary['n_compounds'].max() / fold_summary['n_compounds'].min())
add_check('SPLIT010', 'Development各CV fold规模比不超过1.10', fold_size_ratio <= 1.10, f'{fold_size_ratio:.4f}')
add_check('SPLIT011', 'Development与Novel无相同canonical SMILES', set(development['canonical_smiles'].dropna()).isdisjoint(set(novel['canonical_smiles'].dropna())))
add_check('SPLIT012', 'Novel表示碰撞已逐化合物记录', len(novel_applicability) == len(novel), f'{len(novel_applicability)}/{len(novel)}')
add_check('SPLIT013', 'Development每个化合物均有跨fold相似度适用域记录', len(development_cv_applicability) == len(development), f'{len(development_cv_applicability)}/{len(development)}')
leakage_checks = pd.DataFrame(checks)
assert leakage_checks['status'].eq('PASS').all(), leakage_checks[leakage_checks['status'].eq('FAIL')]

# 输出前按稳定键排序，保证名单哈希可复现
manifest = manifest.sort_values('inchikey_2d').reset_index(drop=True)
novel = novel.sort_values('inchikey_2d').reset_index(drop=True)
natural_novel = novel[novel['chemvista_evaluation_role'].eq('natural_novel')].copy()
selected_novel = novel[novel['chemvista_evaluation_role'].eq('selected_novel')].copy()
transfer = transfer.sort_values('inchikey_2d').reset_index(drop=True)
development = manifest[manifest['supervised_compound_role'].eq('development')].copy()

outputs = {
    'compound_split_manifest.csv': manifest,
    'chemvista_natural_novel.csv': natural_novel,
    'chemvista_selected_novel.csv': selected_novel,
    'chemvista_novel_all.csv': novel,
    'chemvista_transfer.csv': transfer,
    'development_compounds.csv': development,
    'task_split_summary.csv': task_split_summary,
    'novel_coverage_requirements.csv': coverage,
    'cv_fold_summary.csv': fold_summary,
    'selection_trace.csv': selection_trace,
    'leakage_checks.csv': leakage_checks,
    'novel_structure_applicability.csv': novel_applicability,
    'development_cv_applicability.csv': development_cv_applicability,
}
for filename, frame in outputs.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False, encoding='utf-8-sig')

freeze_manifest = {
    'split_version': 'PBT_MSMS_SPLIT_V2',
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'random_seed': SEED,
    'novel_total_target': NOVEL_TOTAL_TARGET,
    'n_cv_folds': N_CV_FOLDS,
    'cv_grouping': {
        'method': 'Morgan ECFP4 Butina structure clusters',
        'radius': 2,
        'n_bits': 2048,
        'butina_distance_threshold': 0.6,
        'use_chirality': False,
        'murcko_scaffold_standard_retained_for_audit': True,
    },
    'selection_inputs': {name: {'path': str(path.relative_to(ROOT)), 'sha256': sha256_file(path)} for name, path in INPUTS.items()},
    'requirements': all_requirements,
    'counts': {
        'chemvista_natural_novel': len(natural_novel),
        'chemvista_selected_novel': len(selected_novel),
        'chemvista_novel_all': len(novel),
        'chemvista_transfer': len(transfer),
        'development': len(development),
        'novel_exact_morgan_representation_collisions': int(novel_applicability['exact_morgan_representation_collision'].sum()),
        'novel_exact_maccs_representation_collisions': int(novel_applicability['exact_maccs_representation_collision'].sum()),
    },
    'list_hashes': {
        'novel_ik2d_sha256': sha256_values(novel['inchikey_2d']),
        'transfer_ik2d_sha256': sha256_values(transfer['inchikey_2d']),
        'development_ik2d_sha256': sha256_values(development['inchikey_2d']),
    },
    'scientific_semantics': {
        'novel': 'IK2D excluded from all supervised development spectra; evaluates unseen-compound generalization.',
        'transfer': 'Same IK2D may occur in public supervised development; only ChemVista spectra are held out; evaluates measurement-domain transfer.',
        'natural_novel': 'ChemVista property compound absent from the public spectrum index.',
        'novel_selection': 'Label-stratified challenge-set selection frozen before model training; not a random or prevalence-representative ChemVista sample.',
        'representation_collisions': 'Exact Morgan/MACCS feature collisions are documented and require include/exclude sensitivity analysis; they are not IK2D identity leakage.',
    },
    'software': {'python': sys.version, 'executable': sys.executable, 'pandas': pd.__version__, 'numpy': np.__version__, 'rdkit': rdBase.rdkitVersion, 'platform': platform.platform()},
    'qc_status': 'PASS',
}
with (OUTPUT_DIR / 'split_freeze_manifest.json').open('w', encoding='utf-8') as handle:
    json.dump(freeze_manifest, handle, ensure_ascii=False, indent=2)
    handle.write('\n')

print('拆分冻结完成，全部检查通过。')
display(leakage_checks)
display(task_split_summary.pivot(index='task_name', columns='subset', values='n_compounds'))
display(fold_summary)
print('输出文件：')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(' -', path.name, f'({path.stat().st_size / 1024:.1f} KB)')

## 7. 冻结后的使用规则

1. 后续模型开发只能使用 `development_compounds.csv`及其`cv_fold`；
2. `chemvista_novel_all.csv`在所有模型和超参数固定前不得评价；
3. `chemvista_transfer.csv`可用于最终跨实验条件评价，但结果必须明确标记为seen-compound transfer；
4. 本次纠错后版本为V2；任何后续拆分修改都必须再次升级版本并在manifest说明原因。Novel是标签分层挑战集，Transfer只是seen-compound measurement transfer。